In [11]:
version = "REPLACE_PACKAGE_VERSION"

---
# Assignment 1 Part 2: N-gram Language Models (Cont.) (30 pts)

In this assignment, we're going to train an n-gram language model that is able to "imitate" William Shakespeare's writing. 

In [12]:
# Configure nltk

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from mads.lib.path import assets

nltk_data_path = assets.find("nltk_data")
if nltk_data_path not in nltk.data.path:
    nltk.data.path.append(nltk_data_path)

[nltk_data] Downloading package punkt to /voc/work/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /voc/work/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [13]:
# Copy and paste the functions you wrote in Part 1 here and import any libraries necessary
# We have tried a more elegant solution by using
# from ipynb.fs.defs.assignment1_part1 import load_data, build_vocab, build_ngrams
# but it doesn't work with the autograder...

import nltk

def load_data():
    """
    Load Shakespeare's sonnets and return one tokenized list per sonnet line.
    """

    sentences = []

    # load the assigned sonnets text file
    file_path = assets.find("gutenberg/THE_SONNETS.txt")

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            # drop the newline character and surrounding whitespace
            sentence = line.rstrip("\n").strip()

            # skip blank lines
            if sentence == "":
                continue

            # skip numeric sonnet headings like 1, 2, ..., 154
            if sentence.isdigit():
                continue

            # lowercase first, then tokenize the sonnet line
            tokens = nltk.word_tokenize(sentence.lower(), preserve_line=True)

            # each remaining line is one sentence under the assignment definition
            sentences.append(tokens)

    return sentences

def build_vocab(sentences):
    """
    Take a list of sentences and return a vocab
    """

    # use a set because vocabulary should contain each token only once
    vocab = set()

    for sentence in sentences:
        # add every token from each sentence to the vocabulary set
        vocab.update(sentence)

    # include sentence boundary markers needed for n-gram language modeling
    vocab.add("<s>")
    vocab.add("</s>")

    # convert back to a list because the assignment requires a list return value
    return list(vocab)

def build_ngrams(n, sentences):
    """
    Take a list of unpadded sentences and create all n-grams as specified by the argument "n" for each sentence
    """

    all_ngrams = []

    for sentence in sentences:
        # create a padded copy so the original sentence list is not modified in place
        padded_sentence = (
            ["<s>"] * (n - 1)
            + list(sentence)
            + ["</s>"] * (n - 1)
        )

        # generate this sentence's n-grams and store them as a reusable list
        sentence_ngrams = list(nltk.ngrams(padded_sentence, n))
        all_ngrams.append(sentence_ngrams)

    return all_ngrams

## Question 4: Guess the next token (20 pts)

With the help of the three functions you wrote in Part 1, let's first answer the following question as a review on $n$-grams.

Assume we are now working with bi-grams. What is the most likely token that comes after the sequence `<s> <s> <s>`, and how likely? Remember that a bi-gram language model is essentially a first-order Markov Chain. So, what determines the next state in a first-order Markov Chain? 

**Complete the function below to return a `tuple`, where `tuple[0]` is a `str` representing the mostly likely token and `tuple[1]` is a `float` representing its (conditional) probability of being the next token.**

In [14]:
def bigram_next_token(start_tokens=("<s>", ) * 3):
    """
    Take some starting tokens and produce the most likely token that follows under a bi-gram model
    """
    
    from collections import Counter

    # load the sonnet lines as tokenized sentences
    sentences = load_data()

    # build bigrams because a bigram model predicts the next token from only the current token
    bigrams_by_sentence = build_ngrams(2, sentences)

    # in a first-order Markov chain, only the most recent token affects the next-token prediction
    current_token = start_tokens[-1]

    # collect every token that immediately follows the current token in the training data
    following_tokens = []
    for sentence_bigrams in bigrams_by_sentence:
        for first_token, second_token in sentence_bigrams:
            if first_token == current_token:
                following_tokens.append(second_token)

    # choose the most frequent follower and estimate its conditional probability from counts
    token_counts = Counter(following_tokens)
    next_token, token_count = token_counts.most_common(1)[0]
    prob = token_count / len(following_tokens)

    return next_token, float(prob)

In [15]:
# Autograder tests

stu_ans = bigram_next_token(start_tokens=("<s>", ) * 3)

assert isinstance(stu_ans, tuple), "Q4: Your function should return a tuple. "
assert len(stu_ans) == 2, "Q4: Your tuple should have two elements. "
assert isinstance(stu_ans[0], str), "Q4: tuple[0] should be a str. "
assert isinstance(stu_ans[1], float), "Q4: tuple[1] should be a float. "

# Some hidden tests


del stu_ans

## Question 5: Train an $n$-gram language model (10 pts)

Now we are well positioned to start training an $n$-gram language model. We can fit a language model using the `MLE` class from `nltk.lm`. It requires two inputs: a list of all $n$-grams for each sentence and a vocabulary, both of which you have already written a function to build. Now it's time to put them together to work. 

**Complete the function below to return an `nltk.lm.MLE` object representing a trained $n$-gram language model.**

In [16]:
from nltk.lm import MLE

def train_ngram_lm(n):
    """
    Train a n-gram language model as specified by the argument "n"
    """
    
    # initialize the maximum likelihood estimator for the requested n-gram order
    lm = MLE(n)

    # load the sonnet lines, then reuse the Part 1 helpers to prepare model inputs
    sentences = load_data()
    ngrams = build_ngrams(n, sentences)
    vocab = build_vocab(sentences)

    # fit estimates each n-gram probability from observed counts over the vocabulary
    lm.fit(ngrams, vocab)

    return lm

In [17]:
# Autograder tests

stu_n = 4
stu_lm = train_ngram_lm(stu_n)
stu_vocab = build_vocab(load_data())

assert isinstance(stu_lm, nltk.lm.MLE), "Q3b: Your function should return an nltk.lm.MLE object. "

assert hasattr(stu_lm, "vocab") and len(stu_lm.vocab) == len(stu_vocab) + 1, "Q3b: Your language model wasn't trained properly. "

del stu_n, stu_lm, stu_vocab

FINALLY, are you ready to compose sonnets like the real Shakespeare?! We provide some starter code below, but absolutely feel free to modify any parts of it on your own. It'd be interesting to see how the "authenticity" of the sonnets is related to the parameter $n$. Do the sonnets feel more Shakespeare when you increase $n$? 

In [18]:
# Every time it runs, depending on how drunk it is, a different sonnet is written. 
n = 3
num_lines = 14
num_words_per_line = 8
text_seed = ["<s>"] * (n - 1)

lm = train_ngram_lm(n)

sonnet = []
while len(sonnet) < num_lines:
    while True:  # keep generating a line until success
        try:
            line = lm.generate(num_words_per_line, text_seed=text_seed)
        except ValueError:  # the generation is not always successful. need to capture exceptions
            continue
        else:
            line = [x for x in line if x not in ["<s>", "</s>"]]
            sonnet.append(" ".join(line))
            break

# pretty-print your sonnet
print("\n".join(sonnet))

how oft when thou from youth convertest ,
but do not so , i love to
this brand she quenched in a cool well
whose strength ’ s sorrow lends but weak
then were not summer ’ s moiety ,
my sweet ’ st me blind ,
for slander ’ s graces ,
’ gainst my strong infection ,
divert strong minds to the time exchanged ,
for that riches where is my judgment knew
that she thinks me young ,
so thou , thy record never can be
which for their virtue only is their show
that god forbid , that taught the dumb
